# 04 · Data integration, quality control and statistical validation
**Owner:** Chan Jia Yi · **Run this notebook first.** It builds the shared data every other notebook reads.

| Section | What it does | Output |
|---|---|---|
| 1. Integration | Raw CSV → one Parquet with all 10M rows + shared derived columns; sample and anomaly extracts | `data/processed/*.parquet` |
| 2. Quality control | One issues log: what was checked, what was found, the decision and why | `outputs/tables/qc_*.csv` |
| 3. Statistical validation | Tests every Currency / Amount / Business-consumer rule, with effect sizes and a verdict | `outputs/tables/rules/rules_04_core.csv` |

**Definitions used by all notebooks**
- *Country* = the **bank (BIC) country**, read from BIC characters 5–6. It is not a verified customer residence.
- *Local currency* = the currency a country's senders use most, **inferred from this dataset** (no external lookup).
- Nothing is deleted. Anomalies and outliers are kept and flagged. `payments_clean` means "parsed and typed", not "every row is valid".
- Verdict scale: **Confirmed** (≥99% of rows, or the effect matches the rule) · **Strong** (95–99%) · **Partial** (70–95%, or only in a subgroup) · **Rejected**.

### Evidence policy (updated 18 Nov 2026)
- **One verdict system, two kinds of rule** (column `verdict_basis` in `evidence_matrix.csv`):
  - *Accuracy rules* state a deterministic prediction (e.g. "Direct ⇔ 2 banks"). They are graded by the share of tested rows that obey them: `Confirmed` ≥ 99% · `Strong` 95–99% · `Partial` 70–95% · `Weak` below that. The tested denominator and any exclusions are always shown.
  - *Effect rules* state a size, an independence or a distribution (e.g. "×7.4 per business party").
    - `Supported`: the stated effect holds **and** the out-of-sample (holdout) estimate agrees with it, by the criterion written in `holdout_rule`, e.g. overlapping 95% CIs.
    - `Approximate`: the direction is right, but the holdout disagrees, is missing, or the model misses. The reason is in `verdict_note`.
    - `Rejected` or `Inconclusive` otherwise.
- **Causal status** says how far the evidence goes. It describes the evidence, not proof of the generator's code.
  - `Mechanism`: needs a 95% CI, an out-of-sample check, a subgroup check and a practical meaning. Rules lacking any of these are automatically downgraded.
  - `Association`: a real, stable pattern whose cause is not identified.
  - `Structural`: true by how the fields are defined.
  - `Rejected`: the stated prediction fails.
  - `Unresolved`: the tested method found no support but cannot rule the mechanism out.
- **Interpretations** ("consistent with …", "plausibly …") are possible explanations, not proven causes. "No measurable association" is not proof of independence, and "no detectable difference" is not proof that something is random.
- **Local time** uses clock offsets inferred from the CSV under stated assumptions (Monday–Friday week, no summer time, one offset per country). They are not verified time zones, and office-hour results on that clock are partly built into the offsets.
- **READY** (end of notebook 04) means every file comes from the current code and data, and every required piece of evidence is present and consistent. It does not prove that a rule is the generator's actual code.
- **Accounts** are always identified by **(BIC8, account number)**, because 2.4% of receiver account numbers recur at several banks.
- **Data used:** only the organiser CSV. Local currency and each country's clock offset are **inferred from the CSV itself**; the clock offsets come from the training period only. No time-zone database, currency table or GDP data enters any calculation.
- Saved outputs are cleared in this pack. Re-run in the order 04 → 01 → 02 → 03 → 04 before quoting any number.

In [1]:
from pathlib import Path
import os, warnings; warnings.filterwarnings('ignore')
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, PROC = ROOT / 'data' / 'raw', ROOT / 'data' / 'processed'
FIG, TAB = ROOT / 'outputs' / 'figures', ROOT / 'outputs' / 'tables'
RULES = TAB / 'rules'                       # one rules file per notebook -> combined into outputs/tables/rules_log.csv
for d in (RAW, PROC, FIG, TAB, RULES): d.mkdir(parents=True, exist_ok=True)
FULL    = PROC / 'payments_clean.parquet'   # all 10M rows (nothing removed) + derived columns, built by notebook 04
SAMPLE  = PROC / 'sample_20pct.parquet'     # 20% Bernoulli sample (seed 42), no free-text columns, still includes anomalies
ANOM    = PROC / 'anomalies.parquet'        # every labelled anomaly row, all columns
SCHEMA_VERSION = '2'                        # bump when derived columns change -> notebook 04 rebuilds the files

def connect(db=':memory:'):
    con = duckdb.connect(str(db))
    con.execute("SET TimeZone='UTC'")       # never skip: otherwise DuckDB shows laptop time (+08)
    con.execute("SET preserve_insertion_order=false")
    try: con.execute("SET enable_progress_bar=false")   # no progress widgets saved in the notebook ("Could not render ... widget" in viewers)
    except Exception: pass                           # (older setups without ipywidgets show no widget anyway)
    con.execute(f"SET temp_directory='{(PROC / 'duck_tmp').as_posix()}'")   # big joins spill to disk instead of crashing
    if os.environ.get('SWIFT_MEMORY'):  con.execute(f"SET memory_limit='{os.environ['SWIFT_MEMORY']}'")   # e.g. 4GB on an 8 GB laptop
    if os.environ.get('SWIFT_THREADS'): con.execute(f"SET threads={int(os.environ['SWIFT_THREADS'])}")
    return con

In [2]:
con = connect()

In [3]:
def sql(q):
    """Run SQL on the full data and return a pandas DataFrame."""
    return con.execute(q).df()
q = sql
def save_table(df, name):
    """Save a result table to outputs/tables/<name>.csv and return it for display."""
    df.to_csv(TAB / f'{name}.csv', index=False); return df

RULE_COLS = ['id', 'group', 'rule', 'test', 'value', 'verdict', 'evidence_files', 'interpretation']
VERDICTS = 'Confirmed (holds in >=99% of rows / effect matches) · Strong (95-99%) · Partial (70-95% or only in a subgroup) · Weak (<70%) · Rejected (prediction fails)'
def verdict_from_share(x):
    return 'Confirmed' if x >= 0.99 else 'Strong' if x >= 0.95 else 'Partial' if x >= 0.70 else 'Weak'

def combine_rules():
    """Combine standardized teammate rule tables and flag unfinished placeholders.

    Running notebook 04 *first* creates its core rules; re-run the final
    consolidation cell after notebooks 01-03 to capture every teammate's work.
    """
    frames = []
    for f in sorted(RULES.glob('rules_*.csv')):
        d = pd.read_csv(f)
        aliases = {'observed_evidence_table_or_chart': 'evidence_files',
                   'interpretation_real_world': 'interpretation', 'confidence': 'verdict'}
        d = d.rename(columns={a:b for a,b in aliases.items() if a in d and b not in d})
        if 'rule' not in d and {'when_condition', 'then_effect'} <= set(d.columns):
            d['rule'] = d['when_condition'].fillna('') + ' -> ' + d['then_effect'].fillna('')
        if 'value' not in d and 'size' in d: d['value'] = d['size']
        for col in RULE_COLS:
            if col not in d: d[col] = ''
        d['source'] = f.name
        frames.append(d)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=RULE_COLS+['source'])
    required = {'rules_04_core.csv','rules_02.csv','rules_03_anomalies.csv'}
    missing = sorted(required - {f.name for f in RULES.glob('rules_*.csv')})
    dup = out['id'].astype(str).duplicated(keep=False) if len(out) else []
    unfinished = out.astype(str).apply(lambda col: col.str.contains(r'TODO|\?\?|A2-example', case=False, regex=True)).any(axis=1) if len(out) else []
    if missing: print('INTEGRATION INCOMPLETE — missing:', ', '.join(missing))
    if len(out) and any(dup): print('INTEGRATION WARNING — repeated rule IDs:', out.loc[dup,'id'].tolist())
    if len(out) and any(unfinished): print('INTEGRATION WARNING — unresolved template cells:', out.loc[unfinished,'id'].tolist())
    out = out[ RULE_COLS + ['source'] + [c for c in out if c not in RULE_COLS + ['source']] ]
    out.to_csv(TAB/'rules_log.csv', index=False)
    return out

## 1. Data integration
- The raw file is ~5 GB, too large for Excel or plain pandas. DuckDB streams it once into a compressed Parquet (~1.3 GB).
- The text columns contain line breaks inside quotes, so the CSV is read with **explicit quoting and fixed column types** (if types were guessed, one bad row could silently turn a number column into text). If the fast parallel reader fails, it retries single-threaded.
- `DISCLAIMER` (one non-empty value) is printed, then dropped.
- **Derived columns are added once here**, so every notebook uses the same definitions. The first build takes about 5–10 minutes. Later runs reuse the files unless `SCHEMA_VERSION` changes.

In [4]:
COLS = {
    "DISCLAIMER": "VARCHAR", "EPOCH_SECONDS": "BIGINT", "CURRENCY_CODE": "VARCHAR", "USD_AMOUNT": "DOUBLE",
    "ORDERING_BIC8": "VARCHAR", "BENEFICIARY_BIC8": "VARCHAR", "BIC8_PATH": "VARCHAR", "TOTAL_DURATION": "BIGINT",
    "CONTINUITY_QFLAG": "VARCHAR", "CURRENCY_CONVERSION_COUNT": "INTEGER", "PARTICIPANT_COUNT": "INTEGER",
    "TRANSACTION_TYPE_CODE": "VARCHAR", "ORDERING_ACCOUNT": "VARCHAR", "BENEFICIARY_ACCOUNT": "VARCHAR",
    "ORDERING_DETAILS": "VARCHAR", "ORDERING_TOKENS": "VARCHAR", "BENEFICIARY_DETAILS": "VARCHAR",
    "BENEFICIARY_TOKENS": "VARCHAR", "ORDERING_TYPE": "VARCHAR", "BENEFICIARY_TYPE": "VARCHAR", "ANOMALY_TYPE": "VARCHAR",
}
DELIM = ','                                   # change to '\t' only if the file is tab-separated
csvs = sorted(RAW.glob('*.csv'))
assert len(csvs) == 1, f'Put exactly one competition CSV in data/raw/ (found {len(csvs)})'
SRC = csvs[0]; print('raw file:', SRC.name, f'({SRC.stat().st_size/1e9:.2f} GB)')

def csv_source(parallel=True):
    cols = ', '.join(f"'{k}': '{v}'" for k, v in COLS.items())
    return (f"read_csv('{SRC.as_posix()}', header=true, delim='{DELIM}', quote='\"', escape='\"', "
            f"columns={{{cols}}}, max_line_size=10000000, parallel={str(parallel).lower()})")

DERIVED = r"""
  SELECT * EXCLUDE (DISCLAIMER),
    row_number() OVER ()                                          AS rid,          -- stable row id (joins between notebooks)
    to_timestamp(EPOCH_SECONDS)                                   AS ts,           -- UTC
    substr(ORDERING_BIC8, 5, 2)                                   AS o_cc,         -- sender bank country
    substr(BENEFICIARY_BIC8, 5, 2)                                AS b_cc,         -- receiver bank country
    ORDERING_TYPE || '2' || BENEFICIARY_TYPE                      AS pair,         -- B2B / B2C / C2B / C2C
    coalesce(nullif(trim(ANOMALY_TYPE), ''), 'NONE')              AS anomaly,      -- blank label = normal payment
    substr(ORDERING_BIC8, 5, 2) <> substr(BENEFICIARY_BIC8, 5, 2) AS cross_border,
    len(string_split(BIC8_PATH, ';'))                             AS path_len,
    contains(BIC8_PATH, '...')                                    AS path_gap,
    regexp_extract(ORDERING_TOKENS,    '<country_(?:code|name):([A-Z]{2})', 1) AS o_det_cc,   -- country written in the details
    regexp_extract(BENEFICIARY_TOKENS, '<country_(?:code|name):([A-Z]{2})', 1) AS b_det_cc,
    trim(regexp_extract(replace(ORDERING_DETAILS,    chr(13), ''), '^/+([^\n]*)', 1)) AS o_det_acct, -- account line in the details
    trim(regexp_extract(replace(BENEFICIARY_DETAILS, chr(13), ''), '^/+([^\n]*)', 1)) AS b_det_acct,
    regexp_replace(replace(BENEFICIARY_DETAILS, chr(13), ''), '^/[^\n]*\n?', '')      AS b_party_text -- details minus account line
  FROM __SRC__"""

ver_file = PROC / '_schema_version.txt'
if not (FULL.exists() and ver_file.exists() and ver_file.read_text().strip() == SCHEMA_VERSION):
    for f in (FULL, SAMPLE, ANOM): f.unlink(missing_ok=True)          # outdated derived files are rebuilt together
    for parallel in (True, False):
        try:
            src = csv_source(parallel)
            print('DISCLAIMER:', con.execute(f'SELECT DISCLAIMER FROM {src} WHERE DISCLAIMER IS NOT NULL LIMIT 1').fetchone()[0][:400])
            con.execute(f"COPY ({DERIVED.replace('__SRC__', src)}) TO '{FULL.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
            print('Parquet written, parallel =', parallel); break
        except Exception as e:
            print('Load failed with parallel =', parallel, '->', str(e)[:300])
    else:
        raise RuntimeError('CSV could not be parsed. Check DELIM and the file in data/raw/.')
    ver_file.write_text(SCHEMA_VERSION)
print('shared parquet:', FULL.name, f'({FULL.stat().st_size/1e9:.2f} GB)')

raw file: swift_umdac_datathon_2026_fictitious_payment_data.csv (4.95 GB)
shared parquet: payments_clean.parquet (1.71 GB)


In [5]:
# Main view `p` = full data with the shared derived columns (see notebook 04, section 1)
assert FULL.exists() and (PROC / '_schema_version.txt').exists() and (PROC / '_schema_version.txt').read_text().strip() == SCHEMA_VERSION, \
    'Shared data missing or outdated: run notebooks/04_integration_validation.ipynb first'
con.execute(f"CREATE OR REPLACE VIEW p AS SELECT * FROM read_parquet('{FULL.as_posix()}')")
# Local currency of each BIC country = the currency its senders use most (inferred from this dataset, no external lookup)
con.execute("""CREATE OR REPLACE TABLE local_ccy AS
  SELECT o_cc AS cc, arg_max(CURRENCY_CODE, n) AS local_ccy FROM
  (SELECT o_cc, CURRENCY_CODE, count(*) n FROM p GROUP BY ALL) GROUP BY o_cc""")
print(f"{con.execute('SELECT count(*) FROM p').fetchone()[0]:,} payments loaded")

10,000,000 payments loaded


In [6]:
# Parse check: numeric columns must have no NULLs (a bad parse shows up here first)
chk = sql("""SELECT count(*) AS rows, count(EPOCH_SECONDS) AS epoch_ok, count(USD_AMOUNT) AS amount_ok,
  min(ts) AS first_ts, max(ts) AS last_ts, count(DISTINCT CURRENCY_CODE) AS currencies, count(DISTINCT o_cc) AS sender_countries,
  sum((anomaly <> 'NONE')::INT) AS anomaly_rows FROM p""")
assert chk.loc[0, 'rows'] == chk.loc[0, 'epoch_ok'] == chk.loc[0, 'amount_ok'], 'Some rows did not parse: check DELIM / quoting'
display(chk)

# Extracts for the analysis notebooks
if not SAMPLE.exists():
    con.execute(f"""COPY (SELECT * EXCLUDE (ORDERING_DETAILS, ORDERING_TOKENS, BENEFICIARY_DETAILS, BENEFICIARY_TOKENS, b_party_text)
                    FROM p USING SAMPLE 20 PERCENT (bernoulli, 42)) TO '{SAMPLE.as_posix()}' (FORMAT parquet, COMPRESSION zstd)""")
if not ANOM.exists():
    con.execute(f"COPY (SELECT * FROM p WHERE anomaly <> 'NONE') TO '{ANOM.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
for f in (SAMPLE, ANOM):
    rows = con.execute("SELECT count(*) FROM read_parquet('" + f.as_posix() + "')").fetchone()[0]
    print(f.name, f'{rows:,} rows')
sql('DESCRIBE p')[['column_name', 'column_type']].T

,rows,epoch_ok,amount_ok,first_ts,last_ts,currencies,sender_countries,anomaly_rows
0,10000000,10000000,10000000,2023-04-30 11:22:26+00:00,2023-07-31 09:56:01+00:00,155,228,7006.0


sample_20pct.parquet 2,001,857 rows
anomalies.parquet 7,006 rows


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33
column_name,EPOCH_SECONDS,CURRENCY_CODE,USD_AMOUNT,ORDERING_BIC8,BENEFICIARY_BIC8,BIC8_PATH,TOTAL_DURATION,CONTINUITY_QFLAG,CURRENCY_CONVERSION_COUNT,PARTICIPANT_COUNT,TRANSACTION_TYPE_CODE,ORDERING_ACCOUNT,BENEFICIARY_ACCOUNT,ORDERING_DETAILS,ORDERING_TOKENS,BENEFICIARY_DETAILS,BENEFICIARY_TOKENS,ORDERING_TYPE,BENEFICIARY_TYPE,ANOMALY_TYPE,rid,ts,o_cc,b_cc,pair,anomaly,cross_border,path_len,path_gap,o_det_cc,b_det_cc,o_det_acct,b_det_acct,b_party_text
column_type,BIGINT,VARCHAR,DOUBLE,VARCHAR,VARCHAR,VARCHAR,BIGINT,VARCHAR,INTEGER,INTEGER,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,BIGINT,TIMESTAMP WITH TIME ZONE,VARCHAR,VARCHAR,VARCHAR,VARCHAR,BOOLEAN,BIGINT,BOOLEAN,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR


In [7]:
# Evidence toolkit (identical in every notebook): 95% CIs, tests, a time-based train/test split and causal labels
from scipy import stats
EVID_COLS = ['id', 'statistic_ci', 'p_value', 'out_of_sample', 'invariance', 'causal_status', 'practical_significance',
             'holdout_agrees', 'holdout_rule']   # holdout_agrees: does the out-of-sample estimate agree with the stated effect / CI?
CAUSAL = {   # how strongly a rule is a cause-and-effect rule of the generator (used in every evidence_*.csv)
    'Mechanism':   'consistent with a generator rule: holds after controls, out of sample and in every subgroup tested, and a placebo / '
                   'negative control shows no effect (supported by the data, not proof of the generator code)',
    'Structural':  'true by how the fields are built (a definition or an accounting identity), not a behavioural finding',
    'Association': 'real and stable pattern, but the data cannot show why: an unobserved cause is possible',
    'Rejected':    'the stated prediction fails, or the pattern is explained away by a confounder',
    'Unresolved':  'the tested method found no support, but it cannot rule the mechanism out (e.g. matching may miss links)'}
# A rule may be called 'Mechanism' only with ALL of these filled in; notebook 04 downgrades it to 'Association' otherwise
MECHANISM_NEEDS = ['statistic_ci', 'out_of_sample', 'invariance', 'practical_significance']

def wilson(k, n, z=1.96):
    """95% Wilson score interval for a share k/n (stays inside 0-100%, works for 0% and 100%)."""
    if not n: return np.nan, np.nan
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def share_ci(k, n, digits=2):
    """'97.80% [97.78%, 97.82%], n = 1,234' for a share k/n."""
    if not n: return 'n = 0'
    lo, hi = wilson(k, n); f = f'{{:.{digits}%}}'
    return f'{f.format(k / n)} [{f.format(lo)}, {f.format(hi)}], n = {int(n):,}'

def two_prop(k1, n1, k2, n2):
    """Share difference (group 1 - group 2), its 95% CI and a two-sided z-test p-value."""
    if not n1 or not n2: return np.nan, (np.nan, np.nan), np.nan
    p1, p2 = k1 / n1, k2 / n2; pp = (k1 + k2) / (n1 + n2)
    se0 = np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2)); se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    pval = 2 * stats.norm.sf(abs(p1 - p2) / se0) if se0 > 0 else (0.0 if p1 != p2 else 1.0)
    return p1 - p2, (p1 - p2 - 1.96 * se, p1 - p2 + 1.96 * se), pval

def paired_prop(a, b):
    """Two yes/no outcomes measured on the SAME rows (e.g. office hour on local clock vs UTC): McNemar test.
    Returns share(a) - share(b), its 95% CI and the p-value; only the discordant rows carry information."""
    a, b = np.asarray(a, bool), np.asarray(b, bool); n = len(a)
    n10, n01 = int((a & ~b).sum()), int((~a & b).sum()); d = (n10 - n01) / n
    se = np.sqrt(max(n10 + n01 - (n10 - n01) ** 2 / n, 0)) / n
    pval = stats.binomtest(n10, n10 + n01).pvalue if 0 < n10 + n01 < 1000 else (2 * stats.norm.sf(abs(n10 - n01) / np.sqrt(n10 + n01)) if n10 + n01 else 1.0)
    return d, (d - 1.96 * se, d + 1.96 * se), pval

def cramers_v_ct(ct):
    ct = np.asarray(ct, float); chi2 = stats.chi2_contingency(ct)[0]
    return np.sqrt(chi2 / (ct.sum() * (min(ct.shape) - 1)))

def v_ci(ct, reps=200, seed=0):
    """Cramér's V with a 95% CI (multinomial bootstrap of the contingency table)."""
    ct = np.asarray(ct, float); rng = np.random.default_rng(seed); n = int(ct.sum())
    boot = [cramers_v_ct(rng.multinomial(n, (ct / n).ravel()).reshape(ct.shape) + 1e-9) for _ in range(reps)]
    return cramers_v_ct(ct), np.percentile(boot, [2.5, 97.5])

def fmt_p(p):
    if p is None or not np.isfinite(p): return 'p = n/a'
    return 'p < 1e-300' if p == 0 else f'p = {p:.1e}' if p < 1e-3 else f'p = {p:.3f}'

def median_ci(x):
    """Distribution-free 95% CI of a median from order statistics (no bootstrap needed)."""
    x = np.sort(np.asarray(x)); n = len(x)
    if n == 0: return np.nan, np.nan
    h = 1.96 * np.sqrt(n) / 2
    return x[max(0, int(np.floor(n / 2 - h)))], x[min(n - 1, int(np.ceil(n / 2 + h)))]

# Out-of-sample check by TIME: a rule is fitted on the first 2/3 of the data window and re-tested on the last 1/3
T_SPLIT = int(con.execute("SELECT quantile_disc(EPOCH_SECONDS, 2/3) FROM p").fetchall()[0][0])
print('train = before', pd.to_datetime(T_SPLIT, unit='s'), 'UTC; test = after')

import hashlib, json as _json
def code_hash(prefix):
    """Fingerprint of a notebook's CODE cells as saved on disk (outputs ignored): proves which code version made a file."""
    f = sorted((ROOT / 'notebooks').glob(f'{prefix}_*.ipynb'))
    if not f: return ''
    cells = _json.loads(f[0].read_text(encoding='utf-8'))['cells']
    return hashlib.sha1('\n'.join(''.join(c['source']) for c in cells if c['cell_type'] == 'code').encode()).hexdigest()[:12]
DATA_STAMP = f"{FULL.stat().st_size}-{int(FULL.stat().st_mtime)}"      # identifies the data build the results came from

def overlap(a, b):
    """True if two intervals (lo, hi) overlap: used to say a holdout estimate agrees with the full-data estimate."""
    return bool(max(a[0], b[0]) <= min(a[1], b[1]))
def holdout(ok, rule):
    return dict(holdout_agrees=bool(ok), holdout_rule=rule)

def save_evidence(rows, name):
    """rows: dicts with EVID_COLS -> outputs/tables/rules/evidence_<name>.csv (merged into the evidence matrix by notebook 04).
    Each file also records the code version (code_hash) and the data build (data_stamp) it came from."""
    e = pd.DataFrame(rows).reindex(columns=EVID_COLS)
    e['code_hash'], e['data_stamp'] = code_hash(name), DATA_STAMP
    bad = set(e.causal_status.dropna()) - set(CAUSAL)
    assert not bad, f'unknown causal_status {bad}'
    e.to_csv(RULES / f'evidence_{name}.csv', index=False); return e

train = before 2023-06-28 12:22:04 UTC; test = after


### 1b. Sender clock offsets, inferred from the data
The offsets are learned from the **training period only** (the first 2/3 of the time window), so the local-time results in the last 1/3 are a real holdout check. The case allows no external datasets, so local time is **not** taken from a time-zone database. Each country's offset is recovered from its own weekly payment rhythm:
1. Align every country's half-hour-of-week activity profile to a shared template.
2. Anchor the template so that local Monday–Friday vs Saturday–Sunday is sharpest.

Countries with fewer than 500 normal payments in the training period get **no** offset and are left out of every local-time test (notebooks 01 and 02 apply the same rule). The result is one offset per country with 30-minute resolution (half-hour zones such as India come out right). Any comparison with real time zones was an informal check outside this pipeline; it is not evidence for any rule, and no time-zone table enters a calculation.

**What the offsets are and are not:** they are the shifts that best line up each country's weekly activity, under the assumptions above. Results that use them, such as office hours on the local clock (R-BC2), are therefore partly by construction. They are **not** independently verified geographic time zones.

In [8]:
# 1b. Sender clock offsets inferred from the data only (replaces the external time-zone table used before)
# Every country follows the same weekly office rhythm on its own clock. Shift each country's half-hour-of-week activity
# profile until it lines up with the others -> relative offset; then anchor the shared profile so that local Mon-Fri vs
# Sat-Sun is sharpest with the day boundary at local midnight -> absolute offset (assumption: weekend = Saturday + Sunday).
def infer_clock_offsets(hist, min_rows=500, step_h=0.5, iters=5):
    """Sender-country clock offsets (hours vs UTC) from the data only.
    hist: DataFrame cc, b (half-hour-of-week bin, 0 = Monday 00:00 UTC), n.
    1) align every country's weekly activity profile to a common template (circular shift with best correlation);
    2) anchor the template so local Mon-Fri vs Sat-Sun is sharpest with day boundaries at local midnight."""
    B = int(round(168 / step_h)); S_MAX = int(round(14 / step_h))
    piv = hist.pivot_table(index='cc', columns='b', values='n', aggfunc='sum', fill_value=0).reindex(columns=range(B), fill_value=0)
    tot = piv.sum(axis=1); piv = piv[tot >= min_rows]
    H = piv.values / piv.values.sum(axis=1, keepdims=True)
    shifts = np.arange(-S_MAX, S_MAX + 1)
    def best(h, T):
        sc = [np.corrcoef(np.roll(h, s), T)[0, 1] for s in shifts]; k = int(np.argmax(sc)); return shifts[k], sc[k]
    T = H.mean(0); S = np.zeros(len(H), int); R = np.zeros(len(H))
    for _ in range(iters):
        for i in range(len(H)): S[i], R[i] = best(H[i], T)
        T = np.mean([np.roll(H[i], S[i]) for i in range(len(H))], 0)
    per_day = int(round(24 / step_h))
    contrast = lambda prof: prof[:5 * per_day].sum() / 5 - prof[5 * per_day:].sum() / 2
    g = shifts[int(np.argmax([contrast(np.roll(T, s)) for s in shifts]))]
    # np.roll(h, s) moves activity s bins later, so local = UTC + (S + g) bins
    return pd.DataFrame({'cc': piv.index, 'offset_hours': (S + g) * step_h, 'n_rows': tot[piv.index].values,
                         'profile_corr': R.round(3)})

# Fitted on the TRAINING period only (before T_SPLIT), so local-time results in the test period are a real holdout check
hist = sql(f"""SELECT o_cc AS cc, ((EPOCH_SECONDS // 1800 + 144) % 336)::INT AS b, count(*) AS n
               FROM p WHERE anomaly = 'NONE' AND EPOCH_SECONDS < {T_SPLIT} GROUP BY ALL""")   # bin 0 = Monday 00:00 UTC
offs = infer_clock_offsets(hist)
offs.to_csv(PROC / 'clock_offsets.csv', index=False); save_table(offs, 'clock_offsets_inferred')
print(f"{len(offs)} countries got an offset from the training period (< 500 normal payments -> no offset -> left out of ALL local-time tests)")
print(f"profile fit after alignment: median correlation {offs.profile_corr.median():.2f}, lowest {offs.profile_corr.min():.2f}")
print('countries per offset (hours):', offs.offset_hours.value_counts().sort_index().to_dict())

186 countries got an offset from the training period (< 500 normal payments -> no offset -> left out of ALL local-time tests)
profile fit after alignment: median correlation 0.95, lowest 0.68
countries per offset (hours): {-10.0: 2, -6.0: 7, -5.0: 9, -4.0: 18, -3.0: 6, -1.0: 1, 0.0: 16, 1.0: 40, 2.0: 26, 3.0: 14, 3.5: 1, 4.0: 6, 5.0: 5, 5.5: 3, 6.0: 4, 6.5: 1, 7.0: 5, 8.0: 9, 9.0: 3, 10.0: 2, 11.0: 3, 12.0: 3, 13.0: 2}


## 2. Quality control → issues log
One row per check, with what was found, the decision and the reason. This table goes on the QC slide almost as-is.

In [9]:
display(save_table(sql("SUMMARIZE SELECT * EXCLUDE (ORDERING_DETAILS, BENEFICIARY_DETAILS, b_party_text) FROM p"), 'qc_summarize_columns'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,EPOCH_SECONDS,BIGINT,1682853746,1690797361,5119419,1686927373.7817295,2191913.668233301,1685179404,1687034674,1688634761,10000000,0.00
1,CURRENCY_CODE,VARCHAR,AED,ZWL,151,None,None,None,None,None,10000000,0.00
2,USD_AMOUNT,DOUBLE,0.01,15664132213.09,3441855,97204.10203820634,9909715.280074067,638.208520046306,3673.2981303071997,19934.85569936775,10000000,0.00
3,ORDERING_BIC8,VARCHAR,0000ADA0,1136US30,8709,None,None,None,None,None,10000000,0.00
4,BENEFICIARY_BIC8,VARCHAR,0000ADA0,1136US30,8632,None,None,None,None,None,10000000,0.00
5,BIC8_PATH,VARCHAR,0000ADA0;...;...;0000NL20;0023ILI0;0001MQM0,1136US30;0313FRP0;0006LV20,4122510,None,None,None,None,None,10000000,0.00
6,TOTAL_DURATION,BIGINT,2,21649,9468,771.5102084,1228.2022156000257,26,57,1104,10000000,0.00
7,CONTINUITY_QFLAG,VARCHAR,N,Y,2,None,None,None,None,None,10000000,0.00
8,CURRENCY_CONVERSION_COUNT,INTEGER,0,17,18,1.3430237,0.9446476223831902,1,1,2,10000000,0.00
9,PARTICIPANT_COUNT,INTEGER,2,22,19,2.5815715,0.9592645007538888,2,2,3,10000000,0.00


In [10]:
APPENDIX_LABELS = ['ROMANCE_SCAM', 'CEO_SCAM', 'ACCOUNT_REPLACEMENT_SCAM', 'FAT_FINGER_MISTAKE', 'DUPLICATE_PAYMENT']
APPX_SQL = ', '.join(f"'{a}'" for a in APPENDIX_LABELS)
RAWCOLS = ', '.join(c for c in COLS if c != 'DISCLAIMER')
checks = [   # (check, sql returning one value, pass condition on the value, decision, reason)
 ('Total rows', 'SELECT count(*) FROM p', lambda v: v > 0, 'Keep all', 'Baseline count'),
 ('Missing values in any column except ANOMALY_TYPE', 'SELECT ' + ' + '.join(f'count(*) - count("{c}")' for c in COLS if c not in ('DISCLAIMER', 'ANOMALY_TYPE')) + ' FROM p',
      lambda v: v == 0, 'None to fix', 'Every field is filled'),
 ('Normal rows (blank ANOMALY_TYPE)', "SELECT sum((anomaly='NONE')::INT) FROM p", lambda v: True, "Recoded blank to 'NONE'", 'Blank is a label, not missing data'),
 ('Full-row hash duplicate candidates (same 64-bit hash of all raw columns)', f'SELECT count(*) - count(DISTINCT hash({RAWCOLS})) FROM p', lambda v: v == 0,
      'Keep; verified below', 'A matching hash is only a candidate: two different rows can share a hash'),
 ('Verified exact duplicate rows (every raw column equal, checked on the hash candidates)',
      f'SELECT coalesce(sum(c - 1), 0) FROM (SELECT count(*) c FROM p WHERE hash({RAWCOLS}) IN (SELECT hash({RAWCOLS}) FROM p GROUP BY 1 HAVING count(*) > 1) GROUP BY {RAWCOLS} HAVING count(*) > 1)',
      lambda v: v == 0, 'Keep; documented exception', 'Only rows equal in every raw column count as exact duplicates'),
 ('Payments sharing time + accounts + amount + currency', """SELECT coalesce(sum(c), 0) FROM (SELECT count(*) c FROM p
      GROUP BY EPOCH_SECONDS, ORDERING_ACCOUNT, BENEFICIARY_ACCOUNT, USD_AMOUNT, CURRENCY_CODE HAVING count(*) > 1)""", lambda v: v == 0,
      'Keep, review', 'Same 5-field signature; not full-row copies'),
 ('USD_AMOUNT <= 0', 'SELECT sum((USD_AMOUNT <= 0)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Would break log-amount analysis'),
 ('USD_AMOUNT range', "SELECT printf('%.2f to %,.0f', min(USD_AMOUNT), max(USD_AMOUNT)) FROM p", lambda v: True, 'Keep', 'Range check'),
 ('Label of the largest amount', 'SELECT arg_max(anomaly, USD_AMOUNT) FROM p', lambda v: v != 'NONE', '-', 'Extremes should be labelled errors'),
 ('TOTAL_DURATION < 0', 'SELECT sum((TOTAL_DURATION < 0)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Impossible value'),
 ('Timestamp range (UTC)', "SELECT strftime(min(ts), '%Y-%m-%d %H:%M') || ' to ' || strftime(max(ts), '%Y-%m-%d %H:%M') FROM p", lambda v: True, '-', 'Coverage'),
 ('Currency code not 3 capital letters', "SELECT sum((NOT regexp_full_match(CURRENCY_CODE, '[A-Z]{3}'))::INT) FROM p", lambda v: v == 0, 'Flag only', 'Format check'),
 ('BIC8 not 8 characters', 'SELECT sum((length(ORDERING_BIC8) <> 8 OR length(BENEFICIARY_BIC8) <> 8)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Country is read from BIC chars 5-6'),
 ('Party type not B/C', "SELECT sum((ORDERING_TYPE NOT IN ('B','C') OR BENEFICIARY_TYPE NOT IN ('B','C'))::INT) FROM p", lambda v: v == 0, 'Flag only', 'Label check'),
 ("CONTINUITY_QFLAG = N agrees with '...' in path (%)", "SELECT round(avg(((CONTINUITY_QFLAG='N') = path_gap)::INT)*100, 3) FROM p", lambda v: v == 100, '-', 'Appendix definition check'),
 ('Conversions > participants - 1', 'SELECT sum((CURRENCY_CONVERSION_COUNT > PARTICIPANT_COUNT - 1)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Logical consistency'),
 ('Direct with PARTICIPANT_COUNT <> 2', "SELECT sum((TRANSACTION_TYPE_CODE='Direct' AND PARTICIPANT_COUNT<>2)::INT) FROM p", lambda v: v == 0, 'Flag only', 'Definition check'),
 ('ORDERING_BIC8 <> first BIC in path (%)', "SELECT round(avg((ORDERING_BIC8 <> split_part(BIC8_PATH, ';', 1))::INT)*100, 3) FROM p", lambda v: True, 'Keep; routing teammate', 'Possible routing rule'),
 ('BENEFICIARY_BIC8 <> last BIC in path (%)', "SELECT round(avg((BENEFICIARY_BIC8 <> list_last(string_split(BIC8_PATH, ';')))::INT)*100, 3) FROM p", lambda v: True, 'Keep; routing teammate', 'Possible routing rule'),
 ('Anomaly labels not in the appendix', f"SELECT coalesce(string_agg(DISTINCT anomaly, ', '), 'none') FROM p WHERE anomaly NOT IN ({APPX_SQL}, 'NONE')", lambda v: v == 'none', 'Keep; investigate (notebook 03, 4.7)', 'Undocumented label'),
 ('Rows with undocumented labels', f"SELECT count(*) FROM p WHERE anomaly NOT IN ({APPX_SQL}, 'NONE')", lambda v: v == 0, 'Keep; ask mentors', 'Undocumented label'),
 ('Details containing carriage return (\\r)', "SELECT sum((ORDERING_DETAILS LIKE '%' || chr(13) || '%' OR BENEFICIARY_DETAILS LIKE '%' || chr(13) || '%')::INT) FROM p",
      lambda v: True, 'Strip \\r before line-length checks', 'Would inflate line length by 1'),
]
rows = []
for c, s, ok, d, r in checks:
    v = con.execute(s).fetchone()[0]
    rows.append((c, v, 'pass' if ok(v) else 'REVIEW', d, r))
qc = save_table(pd.DataFrame(rows, columns=['check', 'result', 'status', 'decision', 'reason']), 'qc_issues_log')
anom = save_table(sql("SELECT anomaly, count(*) n, round(count(*) * 100.0 / sum(count(*)) OVER (), 4) AS pct FROM p GROUP BY 1 ORDER BY 2 DESC"), 'qc_anomaly_counts')
display(qc); display(anom)

,check,result,status,decision,reason
0,Total rows,10000000,pass,Keep all,Baseline count
1,Missing values in any column except ANOMALY_TYPE,0,pass,None to fix,Every field is filled
2,Normal rows (blank ANOMALY_TYPE),9992994,pass,Recoded blank to 'NONE',"Blank is a label, not missing data"
3,Full-row hash duplicate candidates (same 64-bi...,1,REVIEW,Keep; verified below,A matching hash is only a candidate: two diffe...
4,Verified exact duplicate rows (every raw colum...,1,REVIEW,Keep; documented exception,Only rows equal in every raw column count as e...
5,Payments sharing time + accounts + amount + cu...,2,REVIEW,"Keep, review",Same 5-field signature; not full-row copies
6,USD_AMOUNT <= 0,0,pass,Flag only,Would break log-amount analysis
7,USD_AMOUNT range,"0.01 to 15,664,132,213",pass,Keep,Range check
8,Label of the largest amount,FAT_FINGER_MISTAKE,pass,-,Extremes should be labelled errors
9,TOTAL_DURATION < 0,0,pass,Flag only,Impossible value


,anomaly,n,pct
0,NONE,9992994,99.9299
1,ROMANCE_SCAM,2476,0.0248
2,FAT_FINGER_MISTAKE,1940,0.0194
3,CEO_SCAM,945,0.0095
4,DUPLICATE_PAYMENT,679,0.0068
5,ACCOUNT_REPLACEMENT_SCAM,649,0.0065
6,CUSTOMER_RECALL,317,0.0032


**QC interpretation**
- No row needs to be removed. Every anomaly row stays in the data with its label.
- `CUSTOMER_RECALL` is in the data but not in the official appendix. Ask the mentors on Discord. Notebook 03 (4.7) links these rows to earlier payments.
- `DUPLICATE_PAYMENT` rows are **not** full-row copies, so duplicates are matched by account, amount and time instead (notebook 03, 4.3).
- Passing format and range checks does not make every row realistic. Suspicious rows are analysed, not deleted.

## 3. Statistical validation of the Currency, Amount and Business/consumer rules
**Method.** Each rule is written as a testable statement, then measured on the **full data** (counts and accuracy) or on **normal payments in the 20% sample** (models).
- With 10M rows almost every p-value is ~0, so we report **effect sizes**: accuracy %, ln-effects with 95% intervals, R² and Cramér's V.
- Confounding is controlled by comparing payments **within the same sender country** (country fixed effect).
- The interval is a **cluster bootstrap**: it resamples whole sender accounts, not single payments, because payments from one account are not independent.

In [11]:
# Modelling sample: normal (non-anomaly) payments from the 20% sample
n = pd.read_parquet(SAMPLE, columns=['EPOCH_SECONDS','CURRENCY_CODE','USD_AMOUNT','o_cc','b_cc','ORDERING_TYPE','BENEFICIARY_TYPE',
        'pair','anomaly','ORDERING_BIC8','ORDERING_ACCOUNT','TRANSACTION_TYPE_CODE','CURRENCY_CONVERSION_COUNT','PARTICIPANT_COUNT'])
n = n[n.anomaly == 'NONE'].reset_index(drop=True)
n['la'] = np.log(n.USD_AMOUNT)                                   # ln(amount)
n['ob'] = (n.ORDERING_TYPE == 'B').astype(float)                 # sender is business
n['bb'] = (n.BENEFICIARY_TYPE == 'B').astype(float)              # receiver is business
loc = sql('SELECT * FROM local_ccy').set_index('cc').local_ccy
print(f'{len(n):,} normal payments in the modelling sample')

2,000,487 normal payments in the modelling sample


In [12]:
# Sender-local time from clock offsets INFERRED FROM THE DATA (notebook 04, section 1b): no external time-zone table is used
CLK = PROC / 'clock_offsets.csv'
assert CLK.exists(), 'clock_offsets.csv missing: run notebooks/04_integration_validation.ipynb first'
OFFSET_H = pd.read_csv(CLK, keep_default_na=False, na_values=['']).set_index('cc').offset_hours   # keep_default_na: 'NA' = Namibia
def add_local_time(df, cc_col='o_cc', offsets=None):
    """Adds sender-local hour (lh) and weekday (ld, 0 = Mon). -1 where the country had too few rows to infer an offset."""
    off = df[cc_col].map(OFFSET_H if offsets is None else offsets)
    t = df['EPOCH_SECONDS'].values + (off.fillna(0).values * 3600).astype('int64')
    lh = (t // 3600) % 24; ld = (t // 86400 + 3) % 7              # 1970-01-01 was a Thursday -> +3 makes 0 = Monday
    ok = off.notna().values
    return df.assign(lh=np.where(ok, lh, -1), ld=np.where(ok, ld, -1))
print(f'{len(OFFSET_H)} countries with an inferred clock offset')

186 countries with an inferred clock offset


In [13]:
from scipy import stats
def cramers_v(a, b):
    ct = pd.crosstab(a, b); chi2, p, _, _ = stats.chi2_contingency(ct)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), p
def r2_by(y, g):
    # R² of y around its group means, computed in numpy: pandas groupby().transform('mean') returned
    # R² = 1.000 (receiver country) on the laptop's pandas 3 with the full sample
    y = pd.Series(y).to_numpy(float); c = pd.factorize(pd.Series(g).to_numpy(), use_na_sentinel=False)[0]
    ok = np.isfinite(y); y, c = y[ok], c[ok]
    m = np.bincount(c, y) / np.maximum(np.bincount(c), 1)
    return 1 - ((y - m[c]) ** 2).mean() / y.var(ddof=1)
def within(df, cols, by='o_cc'):          # demean inside each sender country (country fixed effect)
    # numpy group means: pandas groupby().transform('mean') was unreliable on the laptop's pandas 3 (see r2_by)
    c = pd.factorize(df[by].to_numpy(), use_na_sentinel=False)[0]; cnt = np.bincount(c)
    X = df[cols].to_numpy(float); m = np.stack([np.bincount(c, X[:, j]) / cnt for j in range(X.shape[1])], 1)
    return pd.DataFrame(X - m[c], columns=cols, index=df.index)

W = within(n, ['la', 'ob', 'bb'])
X, y = W[['ob', 'bb']].values, W.la.values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
resid = W.la - X @ beta

# Cluster bootstrap over sender accounts (Poisson weights per account; per-account sums make it fast)
acc = pd.factorize(n.ORDERING_BIC8 + '|' + n.ORDERING_ACCOUNT)[0]; K = acc.max() + 1
sxx = np.stack([np.bincount(acc, X[:, i] * X[:, j], K) for i, j in [(0, 0), (0, 1), (1, 1)]], 1)
sxy = np.stack([np.bincount(acc, X[:, i] * y, K) for i in (0, 1)], 1)
rng = np.random.default_rng(0); boot = []
for _ in range(300):
    w = rng.poisson(1.0, K); a, b, c = w @ sxx; d = w @ sxy
    boot.append(np.linalg.solve([[a, b], [b, c]], d))
ci = np.percentile(np.array(boot), [2.5, 97.5], axis=0)
print(f'{K:,} sender accounts; ln effect of business sender / receiver:', beta.round(3), '95% cluster CI:', ci.T.round(3).tolist())

209,274 sender accounts; ln effect of business sender / receiver: [1.999 2.002] 95% cluster CI: [[1.98, 2.019], [1.994, 2.009]]


In [14]:
log = []
def add(rid, group, rule, test, value, verdict, evidence, meaning):
    log.append(dict(id=rid, group=group, rule=rule, test=test, value=value, verdict=verdict, evidence_files=evidence, interpretation=meaning))

# ---- Currency (full data)
c1 = sql("""SELECT count(*) AS tested_rows,
                   avg((CURRENCY_CODE = ls.local_ccy)::INT) sender,
                   avg((CURRENCY_CODE = lf.local_ccy)::INT) first_path,
                   avg((CURRENCY_CODE <> ls.local_ccy AND CURRENCY_CODE = lr.local_ccy)::INT) receiver_only,
                   avg((CURRENCY_CODE = lr.local_ccy)::INT) receiver_any
            FROM p LEFT JOIN local_ccy ls ON p.o_cc = ls.cc LEFT JOIN local_ccy lr ON p.b_cc = lr.cc
                   LEFT JOIN local_ccy lf ON substr(split_part(p.BIC8_PATH, ';', 1), 5, 2) = lf.cc""").iloc[0]
add('R-CCY1', 'Currency', "The payment currency is the local currency of the first bank on the path (usually the ordering bank)",
    'full data; local currency = modal sender currency per BIC country, inferred in-sample (holdout checks in notebook 01)',
    f'first path bank {c1.first_path:.1%}; ordering bank {c1.sender:.1%}; receiver-only {c1.receiver_only:.1%} '
    f'({c1.receiver_any:.1%} match the receiver currency incl. same-currency pairs); N = {int(c1.tested_rows):,}',
    verdict_from_share(c1.first_path), 'ccy1_ordering_vs_first_bank.csv; ccy1_train80_test20.csv; ccy1_currency_choice.png',
    "The bank that sends the payment debits in its home currency; FX happens later along the route")
c2 = sql("""SELECT avg((CURRENCY_CONVERSION_COUNT = (ls.local_ccy <> lr.local_ccy)::INT)::INT) acc
            FROM p LEFT JOIN local_ccy ls ON p.o_cc = ls.cc LEFT JOIN local_ccy lr ON p.b_cc = lr.cc
            WHERE TRANSACTION_TYPE_CODE = 'Direct'""").iloc[0, 0]
add('R-CCY2', 'Currency', 'Direct payments: 1 conversion if sender and receiver local currencies differ, otherwise 0', 'prediction accuracy on all Direct rows',
    f'{c2:.1%}', verdict_from_share(c2), 'ccy2_direct_conversion_rule.png', 'With only two banks, FX happens once, and only between currency areas')
c3 = sql('SELECT avg((CURRENCY_CONVERSION_COUNT <= PARTICIPANT_COUNT - 1)::INT) FROM p').iloc[0, 0]
add('R-CCY3', 'Currency', 'Conversions never exceed the number of hops (participants - 1)', 'share of all rows', f'{c3:.2%}', verdict_from_share(c3),
    'ccy3_conversions_by_hops.csv', 'Each correspondent bank can convert at most once')

# ---- Amount (sample, normal payments)
add('R-AMT1', 'Amount', 'Amounts are approximately log-normal given sender country and party types: the ln residual has near-zero skew and kurtosis and the same spread for every pair', 'skew / excess kurtosis / sd of ln residual (an approximation; not proof of the exact generating distribution)',
    f'skew {stats.skew(resid):.3f}, kurtosis {stats.kurtosis(resid):.3f}, sd {resid.std():.2f}',
    'Confirmed' if abs(stats.skew(resid)) < 0.1 and abs(stats.kurtosis(resid)) < 0.2 else 'Partial', 'amt1_lognormal_by_pair.png',
    'Consistent with payment sizes built from many multiplying factors; the exact distribution is not established')
ok2 = (ci[0] <= 2.0).all() and (ci[1] >= 2.0).all() or (abs(beta - 2) < 0.05).all()
add('R-AMT2', 'Amount', 'Each business party (sender or receiver) multiplies the amount by e^2 = 7.4', 'ln effect within sender country, 95% cluster-bootstrap CI',
    f'sender {beta[0]:.3f} [{ci[0,0]:.3f}, {ci[1,0]:.3f}]; receiver {beta[1]:.3f} [{ci[0,1]:.3f}, {ci[1,1]:.3f}]', 'Confirmed' if ok2 else 'Partial',
    'amt2_business_multiplier.png', 'Businesses pay invoices and payroll; consumers send small personal transfers')
r2s, r2r = r2_by(n.la, n.o_cc), r2_by(pd.Series(resid), n.b_cc)
add('R-AMT3', 'Amount', "The sender's country sets the base amount; the receiver's country adds a negligible amount (extra R² below 0.01)", 'R² sender country vs extra R² of receiver country',
    f'{r2s:.3f} vs {r2r:.4f}', 'Confirmed' if r2s > 0.05 and r2r < 0.01 else 'Partial', 'amt3_country_base_amount.png',
    'The sending BIC country sets the typical payment size; why (e.g. income levels) cannot be tested with the competition data')

# ---- Business / consumer
# R-BC1 on ALL normal rows (the 20% sample hides a tiny association that 10M rows can detect). Rows are not independent:
# one sender account makes many payments, so the naive chi-square p is too small; an account-clustered test is reported too.
bc1 = sql("""SELECT ORDERING_TYPE t, ORDERING_BIC8, ORDERING_ACCOUNT, count(*) n, sum((BENEFICIARY_TYPE = 'B')::INT) k
              FROM p WHERE anomaly = 'NONE' GROUP BY ALL""")
g1 = bc1.groupby('t')[['n', 'k']].sum()
ct1 = np.array([[g1.loc['B', 'k'], g1.loc['B', 'n'] - g1.loc['B', 'k']], [g1.loc['C', 'k'], g1.loc['C', 'n'] - g1.loc['C', 'k']]], float)
chi1, pv, _, _ = stats.chi2_contingency(ct1, correction=False); v = float(np.sqrt(chi1 / ct1.sum()))
pr1 = g1.k / g1.n
var1 = {t_: float((((d_.k - d_.n * pr1[t_]) ** 2).sum()) / g1.loc[t_, 'n'] ** 2) for t_, d_ in bc1.groupby('t')}   # cluster = sender account
p_cl = float(2 * stats.norm.sf(abs(pr1['B'] - pr1['C']) / np.sqrt(var1['B'] + var1['C'])))
add('R-BC1', 'Business/consumer', 'Sender and receiver party types have a negligible association on all 10M rows (consistent with independent assignment; not proof of independent draws)',
    "Cramér's V and P(receiver B) by sender type, all normal rows; chi-square p naive and clustered by sender account",
    f"V = {v:.4f}; P(receiver B) {pr1['B']:.2%} for B vs {pr1['C']:.2%} for C senders; p = {pv:.3f} naive, {p_cl:.3f} account-clustered",
    'Confirmed' if v < 0.01 else 'Rejected', 'bc1_type_independence.png', 'Consistent with separate draws for who pays and who gets paid')
lt = add_local_time(n.sample(min(500_000, len(n)), random_state=1)); lt = lt[lt.lh >= 0]
utc_h = pd.to_datetime(lt.EPOCH_SECONDS, unit='s').dt.hour
work_local, work_utc = lt.lh.between(7, 19).mean(), utc_h.between(7, 19).mean()
vt, _ = cramers_v(lt.lh, lt.ORDERING_TYPE)
add('R-BC2', 'Business/consumer', "On each sender country's inferred clock, payments follow an office-hours rhythm; business and consumer timing is nearly the same", 'share 07:00-19:59 on the data-inferred local clock vs UTC; V(hour, sender type)',
    f'{work_local:.1%} local vs {work_utc:.1%} UTC; V = {vt:.3f}', 'Strong (clock offsets inferred from the data)', 'bc2_local_hour_profile.png',
    'Consistent with payments entered in office hours. The clock offsets were inferred assuming a Mon-Fri week, so the pattern is partly built into them: not verified time zones')
act = sql("""SELECT ORDERING_TYPE t, count(*)::DOUBLE / count(DISTINCT (ORDERING_BIC8, ORDERING_ACCOUNT)) per_acct FROM p GROUP BY 1""").set_index('t').per_acct
add('R-BC3', 'Business/consumer', 'Business sender accounts send more payments than consumer accounts', 'payments per (sender BIC, account) over the whole 3-month window (the ratio depends on window length: about x3.8 within 30 days)',
    f"B {act['B']:.1f} vs C {act['C']:.1f} (x{act['B']/act['C']:.1f})", 'Confirmed', 'bc3_activity_per_account.png',
    'Consistent with companies paying many suppliers and staff (not observed directly)')
fr = sql("""SELECT anomaly, count(*) FILTER (WHERE ORDERING_TYPE='B') b, count(*) FILTER (WHERE ORDERING_TYPE='C') c
            FROM p WHERE anomaly IN ('CEO_SCAM','ROMANCE_SCAM') GROUP BY 1""").set_index('anomaly')
pure = (fr.loc['CEO_SCAM','c'] == 0) and (fr.loc['ROMANCE_SCAM','b'] == 0)
add('R-BC4', 'Business/consumer', 'CEO scams only come from business senders; romance scams only from consumer senders', 'labelled anomalies by sender type (full data)',
    f"CEO: B {fr.loc['CEO_SCAM','b']}, C {fr.loc['CEO_SCAM','c']}; romance: B {fr.loc['ROMANCE_SCAM','b']}, C {fr.loc['ROMANCE_SCAM','c']}",
    'Confirmed' if pure else 'Partial', 'bc4_anomaly_by_pair.csv', 'Each scam targets a specific kind of victim')

# ---- Rejected hypotheses (negative results)
vow = pd.Series(resid).groupby(n.CURRENCY_CODE.str[0].isin(list('AEIOU')).values).mean()
add('N1', 'Amount', 'Handout example: consonant currency codes give 3x amounts', 'ln residual difference consonant - vowel (3x would be +1.099)',
    f'{vow[False]-vow[True]:+.3f}', 'Rejected', '-', 'The handout example is illustrative, not a rule in this dataset')
n['xb'] = (n.o_cc != n.b_cc).astype(float); Wx = within(n, ['la', 'ob', 'bb', 'xb'])
bx = np.linalg.lstsq(Wx[['ob','bb','xb']].values, Wx.la.values, rcond=None)[0][2]
raw = n.groupby('xb').USD_AMOUNT.median()
add('N2', 'Amount', 'Cross-border payments are smaller than domestic ones', 'raw median ratio vs within-country ln effect',
    f'raw x{raw[1]/raw[0]:.2f}; same country and party types x{np.exp(bx):.2f}', 'Rejected (confounder: country mix)', 'amt4_crossborder_confounder.png',
    'Confounding: domestic payments are concentrated in sender countries with higher typical amounts; within the same country and party types the cross-border effect is negligible')
nl = pd.Series(resid).groupby((n.CURRENCY_CODE == n.o_cc.map(loc)).values).mean()
add('N3', 'Amount', 'Paying in a non-local currency changes the amount', 'ln residual difference non-local - local', f'{nl[False]-nl[True]:+.3f}', 'Rejected', '-', '-')
h = pd.to_datetime(n.EPOCH_SECONDS, unit='s')
add('N4', 'Amount', 'Transaction type, hour or weekday changes the amount', 'R² of ln amount',
    f"type {r2_by(n.la, n.TRANSACTION_TYPE_CODE):.4f}, hour {r2_by(n.la, h.dt.hour):.4f}, weekday {r2_by(n.la, h.dt.dayofweek):.4f}", 'Rejected', 'amt0_factor_screening.png', '-')

rules04 = pd.DataFrame(log, columns=RULE_COLS); rules04.to_csv(RULES / 'rules_04_core.csv', index=False)
rules04

,id,group,rule,test,value,verdict,evidence_files,interpretation
0,R-CCY1,Currency,The payment currency is the local currency of ...,full data; local currency = modal sender curre...,first path bank 93.4%; ordering bank 89.4%; re...,Partial,ccy1_ordering_vs_first_bank.csv; ccy1_train80_...,The bank that sends the payment debits in its ...
1,R-CCY2,Currency,Direct payments: 1 conversion if sender and re...,prediction accuracy on all Direct rows,97.8%,Strong,ccy2_direct_conversion_rule.png,"With only two banks, FX happens once, and only..."
2,R-CCY3,Currency,Conversions never exceed the number of hops (p...,share of all rows,100.00%,Confirmed,ccy3_conversions_by_hops.csv,Each correspondent bank can convert at most once
3,R-AMT1,Amount,Amounts are approximately log-normal given sen...,skew / excess kurtosis / sd of ln residual (an...,"skew -0.016, kurtosis 0.024, sd 2.08",Confirmed,amt1_lognormal_by_pair.png,Consistent with payment sizes built from many ...
4,R-AMT2,Amount,Each business party (sender or receiver) multi...,"ln effect within sender country, 95% cluster-b...","sender 1.999 [1.980, 2.019]; receiver 2.002 [1...",Confirmed,amt2_business_multiplier.png,Businesses pay invoices and payroll; consumers...
5,R-AMT3,Amount,The sender's country sets the base amount; the...,R² sender country vs extra R² of receiver country,0.131 vs 0.0003,Confirmed,amt3_country_base_amount.png,The sending BIC country sets the typical payme...
6,R-BC1,Business/consumer,Sender and receiver party types have a negligi...,"Cramér's V and P(receiver B) by sender type, a...",V = 0.0010; P(receiver B) 28.84% for B vs 28.7...,Confirmed,bc1_type_independence.png,Consistent with separate draws for who pays an...
7,R-BC2,Business/consumer,"On each sender country's inferred clock, payme...",share 07:00-19:59 on the data-inferred local c...,77.6% local vs 65.0% UTC; V = 0.025,Strong (clock offsets inferred from the data),bc2_local_hour_profile.png,Consistent with payments entered in office hou...
8,R-BC3,Business/consumer,Business sender accounts send more payments th...,"payments per (sender BIC, account) over the wh...",B 44.8 vs C 9.9 (x4.5),Confirmed,bc3_activity_per_account.png,Consistent with companies paying many supplier...
9,R-BC4,Business/consumer,CEO scams only come from business senders; rom...,labelled anomalies by sender type (full data),"CEO: B 945, C 0; romance: B 0, C 2476",Confirmed,bc4_anomaly_by_pair.csv,Each scam targets a specific kind of victim


**Limits of these tests (short, for the methodology slide)**
- These tests show the rules hold in the supplied data. They can't reveal the generator's code itself.
- Local time uses clock offsets **inferred from the data** (section 1b, 30-min steps, one offset per country). This assumes a Saturday–Sunday weekend and no daylight saving. Offsets *between* countries are pinned down by the data, but the common anchor can be off by about an hour, so read local hours as ±1 h. Offsets are learned on the training period only; countries with < 500 normal training payments get no offset and are left out of the local-time tests in every notebook.
- Local currency is inferred from the same payments it is tested on. Section 5 repeats the currency tests with the local currency learned on the first 2/3 of the time window only.
- With 10M rows almost every difference is "significant", so decisions rest on effect sizes, CIs, out-of-sample results and invariance (section 5).
- **Local time is partly circular:** the offsets are chosen so that each country's weekly rhythm lines up, so any later "office hours on the local clock" result is partly built into them. The holdout (offsets from the first 2/3 of the time window, tested on the last 1/3) and the shuffle placebo show that the offsets are stable and country-specific. They do **not** show that the offsets are true geographic time zones.
- **Window length matters** for counts per account (R-BC3): the 3-month ratio (about ×4.5) is larger than a 30-day ratio (about ×3.8). Holdouts therefore compare equal windows.
- **Unresolved mechanisms (documented, not reconstructed):** how account-replacement scams are generated (A5: no predecessor payment found by exact or first-line name matching) and what the undocumented `CUSTOMER_RECALL` label represents (A6: 314 of 317 rows have no linked payment). Both are kept in the data, reported separately and listed as open questions for the organisers.
- **Verdicts are about the tested data.** `Supported` means the effect and its holdout agree. It does not mean the generator's code is known.


In [15]:
# Combined rules log for the whole team (re-run this cell after any notebook adds rules)
combine_rules()

,id,group,rule,test,value,verdict,evidence_files,interpretation,source
0,R-RT1,Routing,Direct payments involve exactly 2 banks; Indir...,share of rows where (type = Direct) <=> (PARTI...,100.00%,Confirmed,rt1_hops_by_type.csv; rt1_hops_by_type.png,Direct = the two banks hold accounts with each...,rules_02.csv
1,R-RT6,Routing,The number of intermediary banks is geometric:...,total variation distance between observed and ...,q = 0.3677; distance = 0.0001,Confirmed,rt6_intermediaries_geometric.csv; rt1_hops_by_...,Banks use the shortest correspondent chain ava...,rules_02.csv
2,R-RT3,Routing,PARTICIPANT_COUNT = number of BIC entries on t...,share of rows without a path gap where path_le...,100.00%,Confirmed,rt2_path_vs_participants.csv,A bank that handles the payment twice counts t...,rules_02.csv
3,R-RT2,Routing,The path starts at the ordering BIC and ends a...,"share of rows, full data, every routing type",start 95.02%; end 95.01%,Strong,rt2_path_vs_participants.csv; rt2_first_bank_m...,A parent/branch BIC or an outsourced sending/r...,rules_02.csv
4,R-RT5,Routing,Each link has about a 5.0% chance of a continu...,chi-square goodness of fit: observed gap rate ...,"goodness of fit chi-square 13.5 on 8 df, p = 0...",Confirmed,rt5_gap_by_participants.csv,Longer chains have more hand-overs where track...,rules_02.csv
5,R-RT7,Routing,Routing type has negligible measured associati...,"Cramér's V (routing type x pair), full data",V = 0.0024,Confirmed,rt4_type_by_pair.csv,Routing patterns are very similar across B/C p...,rules_02.csv
6,R-RT8,Routing,Domestic payments are routed Covered more ofte...,Covered share domestic vs cross-border; Cramér...,Covered 7.34% vs 3.92%; V = 0.0459,Confirmed,rt4_type_by_cross_border.csv,Counter-intuitive (cover payments are normally...,rules_02.csv
7,R-DUR1,Duration,Duration is set by routing: Direct < Indirect ...,median duration by type and hop count (normal ...,"medians Direct 39s, Indirect 281s, Covered 383...",Confirmed,dur1_by_type_hops.csv; dur1_by_type_hops.png; ...,Plausibly each correspondent bank adds process...,rules_02.csv
8,R-DUR2,Duration,Sender and receiver country each add their own...,R² of ln(duration) on sender + receiver countr...,sender 0.051; receiver 0.046; both 0.091,Confirmed,dur2_factor_screening.csv; dur2_country_speed_...,Plausibly each country's banks have their own ...,rules_02.csv
9,R-DUR3,Duration,Scam payments take longer than normal payments...,median duration ratio vs normal payments with ...,same routing: ACCOUNT_REPLACEMENT_SCAM x5.8; C...,Confirmed,dur4_by_anomaly.csv,Consistent with a screening hold before releas...,rules_02.csv


## 4. Files for teammates
- `data/processed/payments_clean.parquet`: all 10M rows with the shared derived columns (`rid, ts, o_cc, b_cc, pair, anomaly, cross_border, path_len, path_gap, o_det_cc, b_det_cc, o_det_acct, b_det_acct, b_party_text`). Query it with DuckDB. Don't load all rows into pandas.
- `data/processed/sample_20pct.parquet`: for charts and models. It **still contains anomalies**, so filter `anomaly == 'NONE'` for normal-only work.
- `data/processed/anomalies.parquet`: every labelled anomaly with all text columns.
- Put your rules in `outputs/tables/rules/rules_<notebook>.csv` and run `combine_rules()`, which updates `outputs/tables/rules_log.csv`.
- If the derived columns change, bump `SCHEMA_VERSION` in the setup cell (all notebooks), and this notebook rebuilds the files.
- `data/processed/clock_offsets.csv`: sender-country clock offset (hours vs UTC), **inferred from the data** (section 1b). Notebooks 01 and 02 use it for local time. No external time-zone data is used.
- `outputs/tables/rules/evidence_<notebook>.csv`: each notebook's evidence rows (CI, out-of-sample, invariance, causal status, practical significance). Section 5 merges them into `outputs/tables/evidence_matrix.csv`.
- Run order: **04 → 01 → 02 → 03 → 04** (the second run of 04 refreshes `rules_log.csv` and `evidence_matrix.csv`).


## 5. Evidence strength, causal status and data compliance
Each rule in `rules_log.csv` is now backed by four kinds of evidence (one row per rule in `outputs/tables/evidence_matrix.csv`):

| Column | What it shows | How |
|---|---|---|
| `statistic_ci` | size of the effect with a 95% CI | Wilson CI for shares, cluster bootstrap over sender accounts for the amount effects, order-statistic CI for medians |
| `p_value` | is it more than chance? plus placebo / negative-control results | z / chi-square / Fisher / Mann-Whitney / permutation |
| `out_of_sample` | does the rule still hold on data it was not built from? | rule fitted on the first 2/3 of the time window, scored on the last 1/3 |
| `invariance` | is it the same in every subgroup (month, country, routing type ...)? | re-estimate inside each subgroup |
| `causal_status` | Mechanism / Structural / Association / Rejected | see `CAUSAL` in the toolkit cell |
| `practical_significance` | why it matters: USD value, minutes, alerts per 10k payments | computed in each notebook |

**Automatic checks:** `estimate_type` says whether `statistic_ci` holds a real 95% CI or only a point estimate. A rule labelled `Mechanism` that lacks any required evidence is downgraded to `Association`, and the gap is written in `causal_note`. `final_verdict` applies the single verdict system from the evidence policy at the top.

**Reading the p-values:** with millions of rows almost any difference is "significant". The decision is made on the **effect size and its CI**, the out-of-sample result and the invariance check, not on p alone.

In [16]:
# Evidence for the rules that live in this notebook only (R-AMT2 cluster CI, rejected hypotheses N1-N4)
ev04 = [
 dict(id='R-AMT2', statistic_ci=f'sender {beta[0]:.3f} [{ci[0, 0]:.3f}, {ci[1, 0]:.3f}], receiver {beta[1]:.3f} [{ci[0, 1]:.3f}, {ci[1, 1]:.3f}] (cluster bootstrap, {K:,} accounts)'),
 dict(id='N1', statistic_ci=f'{vow[False] - vow[True]:+.3f} ln units (3x would be +1.099)', causal_status='Rejected', practical_significance='the handout example is illustrative only'),
 dict(id='N2', statistic_ci=f'raw x{raw[1] / raw[0]:.2f}; same country + party types x{np.exp(bx):.2f}', causal_status='Rejected',
      practical_significance='confounding example: country mix creates a fake cross-border effect'),
 dict(id='N3', statistic_ci=f'{nl[False] - nl[True]:+.3f} ln units', causal_status='Rejected', practical_significance='-'),
 dict(id='N4', statistic_ci='R² < 0.001 for type, hour and weekday', causal_status='Rejected', practical_significance='-'),
]
save_evidence(ev04, '04')

,id,statistic_ci,p_value,out_of_sample,invariance,causal_status,practical_significance,holdout_agrees,holdout_rule,code_hash,data_stamp
0,R-AMT2,"sender 1.999 [1.980, 2.019], receiver 2.002 [1...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,92eff6a516a3,1708807329-1794819479
1,N1,+0.000 ln units (3x would be +1.099),NaN,NaN,NaN,Rejected,the handout example is illustrative only,NaN,NaN,92eff6a516a3,1708807329-1794819479
2,N2,raw x0.62; same country + party types x1.00,NaN,NaN,NaN,Rejected,confounding example: country mix creates a fak...,NaN,NaN,92eff6a516a3,1708807329-1794819479
3,N3,-0.004 ln units,NaN,NaN,NaN,Rejected,-,NaN,NaN,92eff6a516a3,1708807329-1794819479
4,N4,"R² < 0.001 for type, hour and weekday",NaN,NaN,NaN,Rejected,-,NaN,NaN,92eff6a516a3,1708807329-1794819479


In [17]:
# Evidence matrix: every rule from every notebook + its evidence row(s). Re-run after notebooks 01-03.
rules_all = combine_rules()
ev_files = sorted(RULES.glob('evidence_*.csv'))
ev = pd.concat([pd.read_csv(f).assign(ev_source=f.name) for f in ev_files], ignore_index=True) if ev_files else pd.DataFrame(columns=EVID_COLS)
join_txt = lambda s: ' | '.join(dict.fromkeys(str(x) for x in s if pd.notna(x) and str(x).strip() not in ('', '-')))
ev = ev.reindex(columns=list(dict.fromkeys(list(ev.columns) + EVID_COLS)))   # files from older versions lack the newest columns
ev['holdout_agrees'] = ev.holdout_agrees.map({True: 'True', False: 'False', 'True': 'True', 'False': 'False'})
ev = ev.groupby('id').agg({c: (join_txt if c != 'causal_status' else 'first') for c in EVID_COLS[1:] + ['ev_source']}).reset_index()
em = rules_all.merge(ev, on='id', how='left')
for c in ['statistic_ci', 'out_of_sample', 'invariance', 'practical_significance', 'p_value', 'holdout_agrees', 'holdout_rule']: em[c] = em[c].replace('', np.nan)

# 1) One verdict system. 'accuracy' rules state a deterministic prediction -> graded by the share of rows that obey it.
#    'effect' rules state a size / independence / distribution -> graded by whether the CI and the checks support the stated effect.
ACCURACY_RULES = {'R-CCY1', 'R-CCY2', 'R-CCY3', 'R-RT1', 'R-RT2', 'R-RT3', 'R-FX1', 'R-FX3', 'N-FX2', 'R-BC4',
                  'AC1', 'AC2', 'AC3', 'D1', 'D2', 'D3', 'D4', 'A6'}
VERDICT_RULES = {'accuracy': 'share of tested rows that obey the rule: Confirmed >= 99%, Strong 95-99%, Partial 70-95%, otherwise Weak',
                 'effect': 'Supported = the stated effect holds AND the holdout estimate agrees with it (rule in holdout_rule); '
                           'Approximate = the holdout disagrees, is missing, or the effect only partly matches; Rejected; Inconclusive'}
def final_verdict(r):
    v = str(r.verdict)
    if v.startswith(('Rejected', 'Inconclusive')): return v.split(' (')[0]
    if r.id in ACCURACY_RULES: return v.split(' (')[0]
    agrees = set(str(r.holdout_agrees).split(' | ')) == {'True'}      # the holdout estimate must AGREE, not just exist
    return 'Supported' if v.startswith(('Confirmed', 'Strong', 'Supported')) and agrees else 'Approximate'
def verdict_note(r):
    if r.verdict_basis != 'effect' or r.final_verdict in ('Rejected', 'Inconclusive'): return ''
    if 'False' in str(r.holdout_agrees): return 'source-reported holdout disagrees: ' + str(r.holdout_rule)
    if str(r.holdout_agrees) in ('', 'nan'): return 'no holdout comparison'
    return ('source-reported holdout agreement; independent numerical audit still required' if r.final_verdict == 'Supported' else 'effect only partly matches the stated rule')
em['verdict_basis'] = np.where(em.id.isin(ACCURACY_RULES), 'accuracy', 'effect')

# 2) 'Mechanism' needs every kind of evidence; otherwise it is downgraded to 'Association' and the gap is named
need = [c for c in MECHANISM_NEEDS if c in em]
gap = em[need].isna()
down = (em.causal_status == 'Mechanism') & gap.any(axis=1)
em['causal_note'] = np.where(down, 'downgraded from Mechanism: missing ' + gap.apply(lambda r: ', '.join(c for c in need if r[c]), axis=1), '')
em.loc[down, 'causal_status'] = 'Association'
em['causal_status'] = em.causal_status.fillna(em.verdict.astype(str).str.startswith('Rejected').map({True: 'Rejected', False: 'Not assessed'}))
em['final_verdict'] = em.apply(final_verdict, axis=1)
em['verdict_note'] = em.apply(verdict_note, axis=1)

# 3) Say whether the statistic is a real interval or only a point estimate
em['estimate_type'] = np.where(em.statistic_ci.astype(str).str.contains(r'\[', regex=True), '95% CI', 'point estimate only')
em.to_csv(TAB / 'evidence_matrix.csv', index=False)
hp = TAB / 'hypotheses_03.csv'
if hp.exists():   # show the final verdict next to notebook 03's test grade, so the two scales cannot be confused
    hy = pd.read_csv(hp); hy['final_verdict'] = hy.id.map(em.set_index('id').final_verdict).fillna('-'); hy.to_csv(hp, index=False)
rl = pd.read_csv(TAB / 'rules_log.csv')                     # slides read the final verdict; the notebook's own grade is kept as verdict_raw
rl = rl.assign(verdict_raw=rl.verdict).drop(columns='verdict').merge(em[['id', 'final_verdict', 'causal_status', 'verdict_basis']], on='id', how='left')
rl = rl.rename(columns={'final_verdict': 'verdict'}); rl[RULE_COLS + [c for c in rl.columns if c not in RULE_COLS]].to_csv(TAB / 'rules_log.csv', index=False)

# 4) Coverage by causal status (what each status requires is in the 'required' column)
REQUIRED = {'Mechanism': MECHANISM_NEEDS, 'Association': ['statistic_ci', 'practical_significance'], 'Structural': ['statistic_ci'],
            'Rejected': ['statistic_ci'], 'Unresolved': ['statistic_ci']}
cov = em.groupby('causal_status').agg(rules=('id', 'size'), with_95ci=('estimate_type', lambda s: (s == '95% CI').mean()),
        out_of_sample=('out_of_sample', lambda s: s.notna().mean()), invariance=('invariance', lambda s: s.notna().mean()),
        practical=('practical_significance', lambda s: s.notna().mean()))
cov['required'] = [', '.join(REQUIRED.get(i, [])) for i in cov.index]
missing_req = [(r.id, c) for _, r in em.iterrows() for c in REQUIRED.get(r.causal_status, []) if pd.isna(r[c])]
save_table(cov.reset_index(), 'evidence_coverage'); display(cov.round(2))
print('downgraded from Mechanism:', em.loc[down, 'id'].tolist() or 'none')
print('required evidence still missing:', missing_req or 'none')
display(pd.crosstab(em.group, em.final_verdict, margins=True))
em[['id', 'group', 'final_verdict', 'verdict_basis', 'causal_status', 'estimate_type', 'holdout_agrees', 'verdict_note']]

,rules,with_95ci,out_of_sample,invariance,practical,required
causal_status,,,,,,
Association,15,0.67,0.80,0.73,1.00,"statistic_ci, practical_significance"
Mechanism,14,1.00,1.00,1.00,1.00,"statistic_ci, out_of_sample, invariance, pract..."
Rejected,7,0.14,0.00,0.00,0.57,statistic_ci
Structural,7,0.29,0.29,0.00,0.29,statistic_ci
Unresolved,2,1.00,0.00,0.00,1.00,statistic_ci


downgraded from Mechanism: none
required evidence still missing: none


final_verdict,Approximate,Confirmed,Inconclusive,Partial,Rejected,Strong,Supported,All
group,,,,,,,,
Accounts,0,2,0,0,0,1,0,3
Amount,0,0,0,0,4,0,3,7
Anomalies,1,1,1,0,0,0,5,8
Business/consumer,0,1,0,0,0,0,3,4
Currency,0,1,0,1,0,1,0,3
Details,1,3,0,1,0,0,0,5
Duration,0,0,0,0,1,0,3,4
FX,0,2,0,0,2,0,0,4
Routing,0,2,0,0,0,1,4,7


,id,group,final_verdict,verdict_basis,causal_status,estimate_type,holdout_agrees,verdict_note
0,R-RT1,Routing,Confirmed,accuracy,Structural,95% CI,NaN,
1,R-RT6,Routing,Supported,effect,Mechanism,95% CI,True,source-reported holdout agreement; independent...
2,R-RT3,Routing,Confirmed,accuracy,Structural,point estimate only,NaN,
3,R-RT2,Routing,Strong,accuracy,Association,95% CI,True,
4,R-RT5,Routing,Supported,effect,Mechanism,95% CI,True,source-reported holdout agreement; independent...
5,R-RT7,Routing,Supported,effect,Mechanism,95% CI,True,source-reported holdout agreement; independent...
6,R-RT8,Routing,Supported,effect,Association,95% CI,True,source-reported holdout agreement; independent...
7,R-DUR1,Duration,Supported,effect,Mechanism,95% CI,True,source-reported holdout agreement; independent...
8,R-DUR2,Duration,Supported,effect,Association,point estimate only,True,source-reported holdout agreement; independent...
9,R-DUR3,Duration,Supported,effect,Association,95% CI,True,source-reported holdout agreement; independent...


### 5.2 External-data compliance audit
Every input is listed with its source. The scan below searches the code of all notebooks for time-zone databases, web/API access, lookup libraries and files from outside the project data folder.

In [18]:
# compliance-scan (this cell is skipped by the scan below)
# External-data compliance: every input the notebooks use, and an automatic scan of the notebook code for external sources
inputs = save_table(pd.DataFrame([
 ('data/raw/*.csv', 'Case dataset (10M fictitious payments)', 'Provided by the organisers', 'Input'),
 ('data/processed/*.parquet', 'Clean copy + derived columns, 20% sample, anomaly extract', 'Built from the case dataset (section 1)', 'Derived'),
 ('local_ccy (in-memory table)', 'Local currency per BIC country', "Inferred: the currency each country's senders use most", 'Derived'),
 ('data/processed/clock_offsets.csv', 'Clock offset (hours vs UTC) per sender country', 'Inferred from the weekly activity rhythm (section 1b)', 'Derived'),
 ('BIC / IBAN layout', 'Country = BIC letters 5-6; IBAN = country + check digits + account', 'Public format standards (ISO 9362, ISO 13616) used as parsing logic; no data table', 'Format knowledge'),
 ('GDP per capita / income levels', 'Not used. If a slide mentions it, it is labelled as outside context with an APA citation, never as a finding', 'Never loaded or used in a calculation', 'Not used'),
], columns=['input', 'what', 'source', 'type']), 'data_compliance_inputs')
import json, re
PATTERNS = {'time-zone database': r'import\s+pytz|from\s+pytz|pytz\.|zoneinfo|tzdata',
            'web / API access':   r'requests\.|urllib|https?://|yfinance|wbdata|pandas_datareader',
            'country / FX lookup library': r'pycountry|forex_python|iso4217',
            'file outside the project data folder': r"read_(?:csv|parquet|excel|json)\(\s*f?['\"](?!\{|['\"]\s*\+)(?![^'\"]*(?:data/|outputs/))"}   # '{...}' or '...' + var = a project path variable
hits = []
for nbf in sorted((ROOT / 'notebooks').glob('*.ipynb')):
    for i, c in enumerate(json.loads(nbf.read_text(encoding='utf-8'))['cells']):
        src = ''.join(c['source'])
        if c['cell_type'] != 'code' or 'compliance-scan' in src: continue
        for what, pat in PATTERNS.items():
            for m in re.finditer(pat, src): hits.append((nbf.name, i, what, m.group(0)))
scan = save_table(pd.DataFrame(hits, columns=['notebook', 'cell', 'finding', 'match']), 'data_compliance_scan')
print('external-data scan:', 'CLEAN: no external data source in any notebook' if scan.empty else f'{len(scan)} finding(s), see below')
display(inputs); scan

external-data scan: CLEAN: no external data source in any notebook


,input,what,source,type
0,data/raw/*.csv,Case dataset (10M fictitious payments),Provided by the organisers,Input
1,data/processed/*.parquet,"Clean copy + derived columns, 20% sample, anom...",Built from the case dataset (section 1),Derived
2,local_ccy (in-memory table),Local currency per BIC country,Inferred: the currency each country's senders ...,Derived
3,data/processed/clock_offsets.csv,Clock offset (hours vs UTC) per sender country,Inferred from the weekly activity rhythm (sect...,Derived
4,BIC / IBAN layout,Country = BIC letters 5-6; IBAN = country + ch...,"Public format standards (ISO 9362, ISO 13616) ...",Format knowledge
5,GDP per capita / income levels,"Not used. If a slide mentions it, it is labell...",Never loaded or used in a calculation,Not used


,notebook,cell,finding,match


In [19]:
# Submission readiness (re-run LAST): files, FRESHNESS, automated checks and HUMAN scientific sign-off.
# Scientific plausibility and the real generating mechanism cannot be certified solely from metadata fields.
# Review the key rule IDs using the actual exported tables/figures, and explicitly set approved=True in the
# generated CSV only after checking the numerical estimate, hypothesis wording, holdout and exception count.
SIGNOFF_FILE = TAB / 'principal_rule_signoff.csv'
PRINCIPAL_IDS = ['R-CCY1', 'R-AMT2', 'R-BC1', 'R-RT5', 'R-RT6', 'R-DUR1', 'R-FX1', 'A1', 'A2', 'AC3', 'D2']
# The file shows each rule's CURRENT value and verdict. A sign-off is tied to the value the reviewer saw:
# if a later run changes that value, the sign-off is reset to False automatically (a note records the change).
SO_COLS = ['id', 'approved', 'reviewer', 'notes', 'value_reviewed', 'verdict_reviewed', 'current_value', 'final_verdict', 'evidence_files']
so = pd.read_csv(SIGNOFF_FILE, dtype=str, keep_default_na=False) if SIGNOFF_FILE.exists() else pd.DataFrame(columns=SO_COLS)
so = so.reindex(columns=SO_COLS).fillna('').drop_duplicates('id').set_index('id').reindex(PRINCIPAL_IDS).fillna('')
so['approved'] = so.approved.replace('', 'False')
yes = lambda x: str(x).strip().lower() in ('true', '1', 'yes')
emi = em.set_index('id')
for x in PRINCIPAL_IDS:
    now = str(emi.value.get(x, 'missing')); seen = so.loc[x, 'value_reviewed'] or so.loc[x, 'current_value']
    vnow = str(emi.final_verdict.get(x, 'missing')); vseen = so.loc[x, 'verdict_reviewed'] or so.loc[x, 'final_verdict']   # verdict shown when approved
    if yes(so.loc[x, 'approved']):
        why = ([f'value changed from "{seen}" to "{now}"'] if seen and seen != now else []) + \
              ([f'final verdict changed from {vseen} to {vnow}'] if vseen and vseen != vnow else [])
        if why:
            so.loc[x, 'notes'] = (so.loc[x, 'notes'] + ' | ' if so.loc[x, 'notes'] else '') + f'sign-off reset {pd.Timestamp.now():%Y-%m-%d %H:%M}: ' + '; '.join(why)
            so.loc[x, ['approved', 'value_reviewed', 'verdict_reviewed']] = ['False', '', '']
        else: so.loc[x, ['value_reviewed', 'verdict_reviewed']] = [now, vnow]
    so.loc[x, ['current_value', 'final_verdict', 'evidence_files']] = [now, str(emi.final_verdict.get(x, 'missing')), str(emi.evidence_files.get(x, ''))]
so.reset_index().to_csv(SIGNOFF_FILE, index=False)
manual_pending = [x for x in PRINCIPAL_IDS if not yes(so.loc[x, 'approved']) or not so.loc[x, 'reviewer'].strip()]

import json as _json
rl = pd.read_csv(TAB / 'rules_log.csv')
placeholder = rl.astype(str).apply(lambda c: c.str.contains(r'TODO|\?\?|-example', regex=True)).any().any()
mech = em[em.causal_status == 'Mechanism']
evs = {k: pd.read_csv(RULES / f'evidence_{k}.csv') for k in ['01', '02', '03', '04'] if (RULES / f'evidence_{k}.csv').exists()}
# Freshness 1: every evidence file was made by the code that is in the notebook NOW (code hash) and from the current data build
stamp = lambda e, col: str(e[col].iloc[0]) if col in e and len(e) else 'missing (file from an older version)'
stale_code = [k for k, e in evs.items() if stamp(e, 'code_hash') != code_hash(k)]
stale_data = [k for k, e in evs.items() if stamp(e, 'data_stamp') != DATA_STAMP]
# Freshness 2: each notebook's rules file was written in the same run as its evidence file (within 15 minutes)
pairs = {'02': 'rules_02.csv', '03': 'rules_03_anomalies.csv'}
apart = [k for k, f in pairs.items() if not (RULES / f).exists() or abs((RULES / f).stat().st_mtime - (RULES / f'evidence_{k}.csv').stat().st_mtime) > 900]
# Freshness 3: notebooks 01-03 were saved with their executed outputs (04 is being executed right now)
def unrun(prefix):
    f = sorted((ROOT / 'notebooks').glob(f'{prefix}_*.ipynb'))[0]
    return sum(c['cell_type'] == 'code' and c.get('execution_count') is None for c in _json.loads(f.read_text(encoding='utf-8'))['cells'])
not_executed = [k for k in ['01', '02', '03'] if unrun(k)]
# Validity: no missing numbers, every 'Supported' effect has an agreeing holdout, accuracy grades match the share scale
txt_cols = ['statistic_ci', 'out_of_sample', 'invariance', 'p_value']
nan_txt = em[em[txt_cols].fillna('').astype(str).apply(lambda c: c.str.contains(r'\bnan\b|\bn/a\b', regex=True)).any(axis=1)].id.tolist()   # empty fields are not 'nan'
bad_supported = em[(em.final_verdict == 'Supported') & ~em.holdout_agrees.astype(str).eq('True')].id.tolist()
readiness = save_table(pd.DataFrame([
    ('Rule files from notebooks 02, 03, 04 exist', all((RULES / f).exists() for f in ['rules_04_core.csv', 'rules_02.csv', 'rules_03_anomalies.csv'])),
    ('Evidence files from notebooks 01, 02, 03 and 04 exist', len(evs) == 4),
    (f'Evidence made by the CURRENT notebook code (stale: {stale_code or "none"})', len(evs) == 4 and not stale_code),
    (f'Evidence made from the CURRENT data build (stale: {stale_data or "none"})', len(evs) == 4 and not stale_data),
    (f'Rules and evidence written in the same run (not: {apart or "none"})', not apart),
    (f'Notebooks 01-03 saved with executed outputs (not: {not_executed or "none"})', not not_executed),
    ('No placeholder / template text in rules_log.csv', not placeholder),
    ('No repeated rule IDs', not rl.id.astype(str).duplicated().any()),
    ('Every rule has a causal status', 'Not assessed' not in set(em.causal_status)),
    (f'Every Mechanism rule ({len(mech)}) has a 95% CI, out-of-sample, subgroup and practical evidence',
        bool(len(mech)) and mech[MECHANISM_NEEDS].notna().all().all() and (mech.estimate_type == '95% CI').all()),
    ('Every Association rule has a statistic and a practical meaning', not [1 for i, c in missing_req if i in set(em[em.causal_status == 'Association'].id)]),
    (f'Every Supported effect rule has a holdout that agrees (not: {bad_supported or "none"})', not bad_supported),
    (f'No missing numbers (nan) in the evidence text (found in: {nan_txt or "none"})', not nan_txt),
    ('Every rule has a verdict from the single verdict system', em.final_verdict.isin(['Confirmed', 'Strong', 'Partial', 'Weak', 'Supported', 'Approximate', 'Rejected', 'Inconclusive']).all()),
    ('External-data scan clean', scan.empty),
    (f'Principal reconstructed rules reviewed against numerical tables, figures and holdout (pending: {manual_pending or "none"})', not manual_pending),
], columns=['item', 'complete']), 'submission_readiness')
auto_ok = bool(readiness.complete.iloc[:-1].all())        # every row except the human sign-off
print('Automated checks:', 'PASS' if auto_ok else 'FAIL (fix the False rows)')
print('Human sign-off:', 'complete' if not manual_pending else f'{len(manual_pending)} of {len(PRINCIPAL_IDS)} rules pending in principal_rule_signoff.csv')
print('READY (automated checks and reviewer signoff complete)' if readiness.complete.all() else 'NOT READY: fix False rows, including manual numerical rule review')
print('Note: READY means files, code version, data and evidence checks pass. It cannot prove that a rule is the generator\'s real code.')
readiness

Automated checks: PASS
Human sign-off: 3 of 11 rules pending in principal_rule_signoff.csv
NOT READY: fix False rows, including manual numerical rule review
Note: READY means files, code version, data and evidence checks pass. It cannot prove that a rule is the generator's real code.


,item,complete
0,"Rule files from notebooks 02, 03, 04 exist",True
1,"Evidence files from notebooks 01, 02, 03 and 0...",True
2,Evidence made by the CURRENT notebook code (st...,True
3,Evidence made from the CURRENT data build (sta...,True
4,Rules and evidence written in the same run (no...,True
5,Notebooks 01-03 saved with executed outputs (n...,True
6,No placeholder / template text in rules_log.csv,True
7,No repeated rule IDs,True
8,Every rule has a causal status,True
9,"Every Mechanism rule (14) has a 95% CI, out-of...",True


In [20]:
# Findings files, GENERATED from the evidence matrix at the end of every run (never edit them by hand: they would go stale)
#   outputs/tables/FINDINGS.md                      every rule, all feature groups
#   outputs/tables/FINDINGS_currency_amount_BC.md   the Currency / Amount / Business-consumer part (same content, kept for the team)
cell = lambda x: '' if pd.isna(x) else str(x).replace('|', '/').replace('\n', ' ').strip()
cut = lambda x, k=180: (cell(x)[:k] + '…') if len(cell(x)) > k else cell(x)
GROUPS = ['Currency', 'Amount', 'Business/consumer', 'Routing', 'Duration', 'FX', 'Anomalies', 'Accounts', 'Details']
def findings_md(groups, title):
    rd_ok = bool(readiness.complete.all())
    so_txt = 'complete' if not manual_pending else f'{len(manual_pending)} of {len(PRINCIPAL_IDS)} key rules pending'
    L = [f'# {title}', '',
         f'*Generated by notebook 04 on {pd.Timestamp.now():%Y-%m-%d %H:%M} from the full data (10M rows). '
         f'Readiness: **{"READY" if rd_ok else "NOT READY"}** (automated checks: {"PASS" if auto_ok else "FAIL"}; human sign-off: {so_txt}). '
         'Do not edit by hand: re-run the notebooks instead.*', '',
         '**How to read:** *Verdict*: accuracy rules are graded Confirmed (≥99% of rows) / Strong (95–99%) / Partial (70–95%) / Weak; '
         'effect rules are Supported (effect holds and the source notebook reports an agreeing holdout) / Approximate / Rejected / Inconclusive. '
         '*Causal status*: Mechanism (consistent with a generator rule: CI, holdout, subgroup and practical evidence all present) / '
         'Association (real pattern, cause not identified) / Structural (true by definition) / Rejected / Unresolved. '
         'None of these proves the generator\'s actual code.', '']
    for g in groups:
        d = em[em.group == g]
        if d.empty: continue
        L += [f'## {g}', '', '| ID | Rule | Result | Verdict | Causal status | Out of sample | Why it matters |', '|---|---|---|---|---|---|---|']
        for _, r in d.iterrows():
            L.append(f'| {r.id} | {cut(r.rule, 160)} | {cut(r.value, 140)} | {cell(r.final_verdict)} | {cell(r.causal_status)} | '
                     f'{cut(r.out_of_sample, 120)} | {cut(r.practical_significance, 140)} |')
        L.append('')
    L += ['## Limits', '',
          '- Local time uses clock offsets inferred from the data (Mon–Fri week, no summer time, one offset per country): not verified time zones, '
          'and office-hour results on that clock are partly built into the offsets.',
          '- A5 (account-replacement mechanism) and the undocumented CUSTOMER_RECALL label are unresolved and kept as open questions.',
          '- Counts per account depend on the window length (R-BC3); holdouts compare equal windows.',
          f'- Quality control: {int(qc.loc[qc.check.str.startswith("Verified exact"), "result"].iloc[0])} verified exact duplicate row(s); '
          f'{int(qc.loc[qc.check == "Rows with undocumented labels", "result"].iloc[0])} rows with a label outside the appendix. Both are kept and reported.',
          '- Only the organiser CSV is used (see data_compliance_inputs.csv).', '']
    return '\n'.join(L)
(TAB / 'FINDINGS.md').write_text(findings_md(GROUPS, 'Findings: all rules (C2O, UMDAC Datathon 2026)'), encoding='utf-8')
(TAB / 'FINDINGS_currency_amount_BC.md').write_text(findings_md(GROUPS[:3], 'Findings: Currency, USD amount, Business vs consumer'), encoding='utf-8')
print('written: FINDINGS.md and FINDINGS_currency_amount_BC.md')

written: FINDINGS.md and FINDINGS_currency_amount_BC.md
